# Kazakh Profanity Detection — End-to-End Project

**DSSC5113 Bonus Project**  
Binary classification: `0 = no profanity`, `1 = profanity present`.

This notebook covers dataset inspection, split checks, model training, validation-based selection, one final test evaluation, error analysis, and custom inference. The model is word + character TF-IDF with Logistic Regression.

## 1. Environment and reproducibility

Run from the repository root. In Colab, upload/extract the project ZIP first or upload the dataset CSV when prompted.

In [ ]:
import os, json, random, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Python:", sys.version)
print("scikit-learn:", sklearn.__version__)
print("Seed:", SEED)

## 2. Load and inspect the labeled dataset

In [ ]:
data_path = Path("data/kazakh_profanity_dataset.csv")
if not data_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        data_path = Path(next(iter(uploaded)))
    except Exception as e:
        raise FileNotFoundError("Place data/kazakh_profanity_dataset.csv in the project folder or upload the CSV.") from e

df = pd.read_csv(data_path)
print("Shape:", df.shape)
print("Missing values:\n", df.isna().sum())
print("\nLabels:\n", df.label.value_counts().sort_index())
print("\nSplits:\n", df.split.value_counts())
print("\nSplit by label:\n", pd.crosstab(df.split, df.label))
display(df.head())

## 3. Class distribution and text length

In [ ]:
df["text_length"] = df.text.astype(str).str.len()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df.label.value_counts().sort_index().plot(kind="bar", ax=axes[0], title="Class distribution")
axes[0].set_xlabel("Label (0 = clean, 1 = profanity)")
axes[0].set_ylabel("Reviews")
df.text_length.plot(kind="hist", bins=40, ax=axes[1], title="Text length distribution")
axes[1].set_xlabel("Characters")
plt.tight_layout()
plt.show()

## 4. Split leakage check

The curated split was preserved. Exact normalized cross-split duplicate checks found no duplicates. Character TF-IDF near-duplicate checks found no pairs with cosine similarity >= 0.90. At a threshold of >= 0.85, a small number of similar pairs were identified and documented in the leakage audit.

In [ ]:
import re
normalized = (df.text.astype(str).str.lower()
              .str.replace(r"[^\w\s]", " ", regex=True)
              .str.replace(r"\s+", " ", regex=True).str.strip())
tmp = df.assign(normalized_text=normalized)
exact_cross = tmp.groupby("normalized_text").filter(lambda g: g.split.nunique() > 1)
print("Exact cross-split duplicate rows:", len(exact_cross))

## 5. Train and select on Validation

Word n-grams capture lexical expressions. Character n-grams help with misspellings, mixed scripts, and transliteration. TF-IDF is fitted only on Train. We compare `C` values and choose the decision threshold using Validation F1.

In [ ]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, precision_recall_fscore_support

train = df[df.split == "train"]
val = df[df.split == "val"]
test = df[df.split == "test"]

def make_model(C):
    features = FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2,
                                 max_features=50000, sublinear_tf=True,
                                 token_pattern=r"(?u)\b\w+\b")),
        ("char", TfidfVectorizer(analyzer="char", ngram_range=(2, 5), min_df=2,
                                 max_features=150000, sublinear_tf=True))
    ])
    return Pipeline([("features", features),
                     ("classifier", LogisticRegression(C=C, max_iter=3000,
                                                        solver="liblinear", random_state=SEED))])

def choose_threshold(y_true, prob):
    precision, recall, thresholds = precision_recall_curve(y_true, prob)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    idx = int(np.nanargmax(f1[:-1]))
    return float(thresholds[idx]), float(f1[idx])

candidates, fitted = [], {}
for C_value in [0.1, 0.3, 1.0, 3.0, 10.0]:
    model = make_model(C_value)
    model.fit(train.text.astype(str), train.label.astype(int))
    val_prob = model.predict_proba(val.text.astype(str))[:, 1]
    threshold, val_f1 = choose_threshold(val.label.astype(int), val_prob)
    val_pred = (val_prob >= threshold).astype(int)
    p, r, f, _ = precision_recall_fscore_support(val.label, val_pred, average="binary", zero_division=0)
    candidates.append({"C": C_value, "threshold": threshold,
                       "precision": p, "recall": r, "f1": f})
    fitted[C_value] = model

selection = pd.DataFrame(candidates).sort_values(["f1", "recall", "precision"], ascending=False)
display(selection)
best = selection.iloc[0]
selected_C = float(best.C)
threshold = float(best.threshold)
model = fitted[selected_C]
print("Selected C:", selected_C)
print("Selected threshold:", threshold)

## 6. Final held-out Test evaluation

Do not change model settings after inspecting these results. Test is used once for the final evaluation.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_test = test.text.astype(str)
y_test = test.label.astype(int).to_numpy()
test_prob = model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= threshold).astype(int)
print("Test size:", len(y_test))
print("Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print(classification_report(y_test, test_pred, digits=4, zero_division=0))
cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
print("Confusion matrix [[TN, FP], [FN, TP]]:\n", cm)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm)
ax.set_xticks([0, 1], ["Predicted 0", "Predicted 1"])
ax.set_yticks([0, 1], ["Actual 0", "Actual 1"])
for i in range(2):
    for j in range(2): ax.text(j, i, str(cm[i, j]), ha="center", va="center")
ax.set_title("Test confusion matrix")
plt.tight_layout(); plt.show()

## 7. Inspect mistakes

False positive: a clean comment is flagged as profane. False negative: a profane comment is missed. These examples illustrate context and spelling limitations.

In [ ]:
test_results = test[[c for c in ["kazsandra_id", "text", "label"] if c in test.columns]].copy()
test_results["predicted_label"] = test_pred
test_results["profanity_probability"] = test_prob
test_results["correct"] = test_results.label.to_numpy() == test_pred
errors = test_results[~test_results.correct]
print("Number of test errors:", len(errors))
display(errors.head(12))

## 8. Save model and configuration

In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/kazakh_profanity_tfidf_logreg.joblib")
config = {"model": "word + character TF-IDF + Logistic Regression",
          "seed": SEED, "selected_C": selected_C, "threshold": threshold,
          "train_size": len(train), "validation_size": len(val), "test_size": len(test),
          "sklearn_version": sklearn.__version__}
with open("models/config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print("Saved model and config.")

## 9. Predict a new custom comment

In [ ]:
def predict_text(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Enter a non-empty comment.")
    probability = float(model.predict_proba([text])[0, 1])
    label = int(probability >= threshold)
    return {"label": label,
            "prediction": "profanity present" if label else "no profanity detected",
            "profanity_probability": round(probability, 4),
            "threshold": round(threshold, 4)}

predict_text("Кымбаттап кетипти говно")

The final model is a reproducible baseline. It reaches test Accuracy 0.8893 and profanity-class F1 0.9062 on 271 held-out comments. The refreshed model uses C=3.0 and threshold=0.4453. The main limitations are contextual ambiguity (`мал` can mean livestock), non-standard Latin transliteration, rare spelling variants, and the limited, manually curated dataset. The probability score is not guaranteed to be calibrated.